# Etapa 5n — Cierre de la versión 2 del algoritmo

**Qué es.** La versión 2 junta las mejoras que quedaron de la ronda para la entrega final (desvío 14.11 del plan), decididas solo con desarrollo y con la regla escrita antes de correr:

| Etapa | Cambio | Matthews en desarrollo (validación cruzada) |
|---|---|---|
| — | Versión 1, congelada (`congelado-test`): resta del fondo, 26 mapas | 0,7302 |
| 5g | Normalización del fondo por división | 0,7509 |
| 5k | Wiener con el ruido estimado (por robustez: con ruido 10, de 0,648 a 0,712) | 0,7525 (con 5h y 5i) |
| 5m | Regresión con L1 (usa la mitad de los mapas) | 0,7502 (con 5h, 5i y 5l) |
| 5o | Versión liviana: sin relleno de exudados (5i), aperturas por reconstrucción (5h) ni B-COSFIRE (5l), por costo | **0,7484**, 13,4 s por foto (la versión 1, 17,5 s) |

No se adoptaron: la histéresis (5j), el borde asimétrico (5h), la simetría de fase, las aperturas por caminos y el hessiano (5l). El relleno, las aperturas y B-COSFIRE se adoptaron en su ronda y se sacaron en `05o` por costo (desvío de criterio declarado).

**Cómo se ajusta** (como la versión 1 en la etapa 8, con todas las imágenes de desarrollo):
1. Píxeles de entrenamiento de las 30 imágenes (entrenamiento equilibrado por grosor) con los 26 mapas de la versión 1, calculados sobre el verde con Wiener con ruido estimado y división del fondo.
2. Regresión con L1 (`pipeline.fit_weights_l1`): λ por validación cruzada con los 5 grupos de personas y la regla de un error estándar, y ajuste con las 30 imágenes. Se guardan solo los mapas con peso distinto de cero; los demás no se calculan.
3. Valor fijo del umbral con las 30 imágenes (`choose_value`), y el de la referencia clásica (detector de líneas solo) con la misma regla.
4. Se guarda en `models/pesos_v2.json`, con su preprocesamiento y su lista de mapas. **La versión 1 (`models/pesos_combinacion.json`) no cambia y sigue siendo la oficial:** es la que se evaluó en prueba.

**Qué estimación vale.** La de validación cruzada de `05o` (Matthews 0,748, vasos finos 32,7 %, Dice de línea central 0,750; con ruido 10, 0,721), con la salvedad de que es optimista: varias rondas se decidieron con las mismas 30 imágenes. Medir esta versión con las imágenes con que se ajustó no dice nada nuevo, así que acá solo se comprueba que funciona. El conjunto de prueba no se usa.

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_image, load_split
from src.evaluation import map_images
from src.pipeline import (WEIGHTS_PATH, WEIGHTS_V2_PATH, choose_value, fit_weights_l1, load_feature_names,
                          save_reference_value, save_weights, segment_final)
from src.split import person_folds

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
MODE = C.SEGMENT_BACKGROUND_MODE
print(MODE, "|", len(C.SEGMENT_FEATURES), "mapas candidatos | ajuste:", C.SEGMENT_FIT)

división + Wiener con ruido estimado | 26 mapas candidatos | ajuste: L1


## Píxeles de entrenamiento de las 30 imágenes

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id, background=MODE)

t = time.time()
samples = dict(map_images(pass1, ids, workers=4))
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 151 s


## Ajuste con L1 y valores fijos

In [3]:
t = time.time()
w, lam, info = fit_weights_l1(samples, ids, {i: int(folds[i]) for i in ids})
print(f"Ajuste: {time.time() - t:.0f} s | λ = {lam:.5f} (el de menor devianza: {info['best']:.5f})")
keep = np.flatnonzero(w[1:])
names = [C.SEGMENT_FEATURES[j] for j in keep]
w_v2 = np.r_[w[0], w[1:][keep]]
sub = {i: {k: (v[:, keep] if k in ("x", "x_thr") else v) for k, v in s.items()} for i, s in samples.items()}
value = choose_value(sub, w_v2, ids)
ref_value = choose_value(samples, None, ids)
print(f"{len(names)} mapas | valor fijo {value:.4f} | referencia {ref_value:.4f}")
pd.Series(w_v2[1:], index=names, name="peso").sort_values(key=np.abs, ascending=False).round(3)

Ajuste: 203 s | λ = 0.00241 (el de menor devianza: 0.00005)


13 mapas | valor fijo 0.8591 | referencia 1.0095


detector de líneas 24    1.102
filtro adaptado 8        1.026
Frangi 16               -0.784
detector de líneas 16    0.525
Frangi 8                -0.520
filtro adaptado 4        0.382
Gabor                    0.263
detector de líneas 8     0.206
Gabor 24                 0.196
sombrero de copa         0.157
Frangi 2                -0.103
verde corregido          0.076
Gabor 16                 0.013
Name: peso, dtype: float64

## Se guarda en `models/pesos_v2.json`

In [4]:
save_weights(w_v2, WEIGHTS_V2_PATH, value=value, background=MODE, feature_names=names,
             note=("Versión 2 liviana (etapas 5g a 5o, desvío 14.11): Wiener con ruido estimado y división del "
                   "fondo; regresión con L1 sobre los 26 mapas de la versión 1, λ por validación cruzada por personas con la regla "
                   f"de un error estándar (λ = {lam:.5f}); ajustada con las 30 imágenes de desarrollo "
                   "(notebooks/05n_version2.ipynb). No se evaluó en prueba."))
save_reference_value(ref_value, WEIGHTS_V2_PATH)
assert load_feature_names(WEIGHTS_V2_PATH) == names
print(WEIGHTS_V2_PATH.read_text(encoding="utf-8")[:600])

{
  "features": [
    "verde corregido",
    "sombrero de copa",
    "Gabor",
    "Frangi 2",
    "filtro adaptado 4",
    "detector de líneas 8",
    "filtro adaptado 8",
    "Frangi 8",
    "detector de líneas 16",
    "Gabor 16",
    "Frangi 16",
    "detector de líneas 24",
    "Gabor 24"
  ],
  "intercept": -1.4607001639628328,
  "coefficients": [
    0.07625563153958487,
    0.15696599626592372,
    0.2628550000605758,
    -0.10322794108098543,
    0.3817086628593885,
    0.20613270168681405,
    1.025737102926821,
    -0.5199017316711326,
    0.5250925033259318,
    0.013151543659183883


## Comprobación: funciona con `segment_final` y cuánto tarda (3 imágenes de desarrollo)

In [5]:
rows = []
for image_id in ids[:3]:
    rgb = load_image(image_id)
    for version, path in (("v1", WEIGHTS_PATH), ("v2", WEIGHTS_V2_PATH)):
        t = time.time()
        m = segment_final(rgb, path=path)
        rows.append({"imagen": image_id, "versión": version, "segundos": round(time.time() - t, 1),
                     "fracción marcada": round(float(m.mean()), 4)})
    t = time.time()
    r = segment_final(rgb, reference=True, path=WEIGHTS_V2_PATH)
    rows.append({"imagen": image_id, "versión": "v2 referencia", "segundos": round(time.time() - t, 1),
                 "fracción marcada": round(float(r.mean()), 4)})
pd.DataFrame(rows)

,imagen,versión,segundos,fracción marcada
0,01_h,v1,18.4,0.0755
1,01_h,v2,13.1,0.0739
2,01_h,v2 referencia,8.6,0.0765
3,02_h,v1,17.9,0.0796
4,02_h,v2,13.2,0.0767
5,02_h,v2 referencia,8.5,0.0799
6,03_h,v1,18.2,0.0775
7,03_h,v2,14.4,0.0766
8,03_h,v2 referencia,8.7,0.0797


## Resumen (escrito después de correr)

**La versión 2 quedó en `models/pesos_v2.json`** y se usa con `segment_final(rgb, path=WEIGHTS_V2_PATH)` o `python scripts/segmentar.py foto.jpg mascara.png --v2`. La versión 1 no cambió.

- **Preprocesamiento:** Wiener con el ruido estimado en cada foto y división del fondo.
- **13 mapas** (λ = 0,00241 por la regla de un error estándar; el de menor devianza era 0,00005, con casi todos los mapas): verde corregido, sombrero de copa; Gabor (mayor respuesta), Gabor 16 y 24; filtro adaptado 4 y 8; detector de líneas 8, 16 y 24; Frangi 2, 8 y 16. Los pesos más grandes son los del detector de líneas 24 (+1,10), el filtro adaptado 8 (+1,03) y Frangi 16 (−0,78).
- **Valor fijo 0,859** sobre la probabilidad (la versión 1 usa 0,857). **Referencia clásica: 1,010** sobre el detector de líneas estandarizado (la versión 1, 0,945).
- **Tarda 13 a 14 segundos por foto, contra 18 de la versión 1**, medido en las mismas fotos y en esta máquina.
- Marca entre el 7,4 % y el 7,7 % de la foto en las tres imágenes de la comprobación, apenas menos que la versión 1.

**Estimación de desarrollo** (validación cruzada, `05o`): Matthews 0,748, vasos finos 32,7 %, Dice de línea central 0,750, contra 0,730 de la versión 1 con el mismo método (`05g`); con ruido de desvío 10, 0,721 (la versión 1 sin Wiener rondaba 0,65 en `08`). Es optimista: varias rondas se decidieron con las mismas 30 imágenes. **No se evaluó en prueba.**